# HELIOS Ablation Notebook

L0-L3 pipeline matrix. All data sourced from frozen OSF artefacts only.

No live inference is performed in this notebook.

In [1]:
import json
from pathlib import Path

_here = Path.cwd()
_root = next(p for p in [_here, *_here.parents] if (p / "pyproject.toml").exists())
OSF_DIR = _root / "research" / "osf"

osf_seeds = json.loads((OSF_DIR / "seeds.json").read_bytes())
thresholds = json.loads((OSF_DIR / "thresholds.json").read_bytes())
variant_hashes = json.loads((OSF_DIR / "variant_hashes.json").read_bytes())
prompt_sha = json.loads((OSF_DIR / "prompt_sha.json").read_bytes())
analysis_plan = json.loads((OSF_DIR / "analysis_plan.json").read_bytes())

print(f"Generated at: {osf_seeds['generated_at_iso']}")
print(f"Schema version: {osf_seeds['schema_version']}")

Generated at: 2026-05-19T03:53:06.670786+00:00
Schema version: v1


## L0 — VCL Flag Matrix

Eight confirmatory variants across 14 feature flags.

In [2]:
import pandas as pd

rows = []
for v in variant_hashes["variants"]:
    row = {"variant": v["name"], "status": v["status"]}
    row.update(v["flags"])
    rows.append(row)

df = pd.DataFrame(rows).set_index("variant")
bool_cols = [c for c in df.columns if c not in ("status", "ingest_mode")]
df[bool_cols] = df[bool_cols].map(lambda x: "Y" if x else "N")
print(f"Variant count: {len(df)}")
print(df.to_string())

Variant count: 8
                                 status acp cbr dpipe dpipe_propagation gpipe ingest_mode l2b_graph l2c_llm lpipe mahc p4_cognitive reconcile router ueg_c_structural
variant                                                                                                                                                              
HELIOS-Full                confirmatory   Y   Y     Y                 Y     Y    recorded         Y       Y     Y    Y            Y         Y      Y                Y
HELIOS-noLLM               confirmatory   Y   Y     Y                 Y     Y    recorded         Y       N     N    Y            Y         Y      Y                Y
HELIOS-noGraph             confirmatory   Y   Y     Y                 Y     N    recorded         N       Y     Y    Y            Y         Y      Y                Y
HELIOS-D                   confirmatory   N   N     Y                 Y     N    recorded         N       N     N    N            Y         N      N     

## L1 — D-pipe Calibration

LOO-CV results from Milestone 2. Thresholds frozen at milestone exit.

In [3]:
dpipe = thresholds["dpipe"]
print("D-pipe Calibration (Milestone 2 LOO-CV)")
print(f"  Calibration incidents: {dpipe['calibration_incidents']}")
print(f"  Frozen at: {dpipe['frozen_at_milestone']}")
print(f"  PPR alpha: {dpipe['ppr_alpha']}")
print(f"  Pruner threshold: {dpipe['pruner_threshold']}")
print(f"  Rho threshold: {dpipe['rho_threshold']}")
v_hr = dpipe["loo_cv_hr_at_3"]
v_cpr = dpipe["loo_cv_cpr"]
print(f"  LOO-CV HR@3: {v_hr if v_hr is not None else 'N/A'}")
print(f"  LOO-CV CpR: {v_cpr if v_cpr is not None else 'N/A'}")
print(f"  Pruner efficacy gate: {dpipe['pruner_efficacy_gate']}")
print(f"  Integrity rate gate: {dpipe['integrity_rate_gate']}")

D-pipe Calibration (Milestone 2 LOO-CV)
  Calibration incidents: 15
  Frozen at: Milestone 2
  PPR alpha: 0.85
  Pruner threshold: 0.02
  Rho threshold: 0.2
  LOO-CV HR@3: 0.533333
  LOO-CV CpR: 0.442222
  Pruner efficacy gate: 0.2
  Integrity rate gate: 0.4


## L2 — G-pipe

PPR disagreement gate calibration from Milestone 3. A-H6 requires sentinel row exclusion (`narrative != 'gpipe-gated-or-skipped'`).

In [4]:
gpipe = thresholds["gpipe"]
print("G-pipe Calibration (Milestone 3 PPR disagreement gate)")
print(f"  Frozen at: {gpipe['frozen_at_milestone']}")
print(f"  Disagreement threshold: {gpipe['disagreement_threshold']}")
print(f"  PPR alpha: {gpipe['ppr_alpha']}")
print(f"  G-pipe LOO-CV HR@3 (held-out): {gpipe['gpipe_hr_at_3_held_out']}")
print(f"  D-pipe LOO-CV HR@3 (held-out): {gpipe['dpipe_hr_at_3_held_out']}")
print(f"  Gate passed: {gpipe['gate_passed']}")
print(f"  Incidents triggered: {gpipe['n_incidents_triggered']}")
print()
a_h6 = next(h for h in analysis_plan["family_a_hypotheses"] if h["id"] == "A-H6")
print(f"A-H6 sentinel filter: {a_h6['filter']}")
print(f"A-H6 alpha: {a_h6['alpha']}")

G-pipe Calibration (Milestone 3 PPR disagreement gate)
  Frozen at: Milestone 3
  Disagreement threshold: 0.02
  PPR alpha: 0.85
  G-pipe LOO-CV HR@3 (held-out): 0.6
  D-pipe LOO-CV HR@3 (held-out): 0.4
  Gate passed: True
  Incidents triggered: 20

A-H6 sentinel filter: narrative != 'gpipe-gated-or-skipped'
A-H6 alpha: 0.0125


## L3 — L-pipe Prompt Governance

Protocol A settings — prompt SHA locked, no live inference in this notebook.

In [5]:
print("L-pipe Prompt Governance (Protocol A, frozen; no live inference)")
print(f"  Prompt version: {prompt_sha['prompt_version']}")
print(f"  Model: {prompt_sha['model_name']}")
print(f"  Prompt SHA-256: {prompt_sha['prompt_sha256']}")
print(f"  Frozen at: {prompt_sha['frozen_at_milestone']}")
print(f"  Note: {prompt_sha['note']}")

L-pipe Prompt Governance (Protocol A, frozen; no live inference)
  Prompt version: rca_v1
  Model: llama3.1:8b
  Prompt SHA-256: 376e555b0bd07a14667c9d5f09275ce5507ddce443fa01781bd7dc40a9365e47
  Frozen at: Milestone 3
  Note: Production target: llama3.1:70b via vLLM. Deviation logged (see deviation_log entries 11-12).


## L4 — Exploratory Statistical Analysis (Milestone 4)

Wilcoxon signed-rank + Holm–Bonferroni correction over A-family hypotheses.
**These are exploratory results on the OTEL corpus. No binding inference is made.**
Confirmatory inference is reserved for Phase 2 (AIOpsLab corpus).

In [ ]:
import json
from pathlib import Path

_results_path = _root / "data" / "m4_results.json"
if _results_path.exists():
    results = json.loads(_results_path.read_text())
    print(f"Corpus: {results['corpus']}")
    print(f"Type: {results['analysis_type']}")
    print(f"Note: {results['note']}")
else:
    print(f"[PENDING] {_results_path} not yet generated.")
    print("Run: python scripts/run_ablation.py + fuse_verdicts.py + analyse_results.py")
    results = None

In [ ]:
if results is not None:
    hyps = results["hypotheses"]
    print(
        f"{'Hyp':<8} {'Comparison':<40} {'raw p':>8} {'holm p':>8} {'r':>6} {'H0 rej':>7}"
    )
    print("-" * 80)
    for hyp_id, data in hyps.items():
        praw = data.get("raw_pvalue", float("nan"))
        pcorr = data.get("corrected_pvalue", float("nan"))
        r = data.get("effect_r", float("nan"))
        rej = data.get("rejected", False)
        cmp = data.get("comparison", "—")
        print(f"{hyp_id:<8} {cmp:<40} {praw:>8.4f} {pcorr:>8.4f} {r:>6.3f} {rej!s:>7}")
else:
    print("[PENDING] Run ablation pipeline to populate results.")

In [ ]:
print("Power disclosure:")
print("  n=20 incidents; Wilcoxon exact two-sided; exploratory alpha=0.05")
print("  Effect sizes r < 0.3 are small; interpretive caution applies.")
print("  Confirmatory inference is reserved for Phase 2 (AIOpsLab corpus).")